# Analyze Oxygen Not Included Like a Data Scientist

A reproducible Python 3.15 walkthrough using **OniHarness**, pandas and Matplotlib.
Connect to a real colony, display its rendered viewport in the notebook, inspect food and duplicant data, then refresh the live view.

## Goal and scope

This notebook observes the loaded game. Every SDK observation verifies pause; no cell resumes the simulation, places orders, spawns resources, or loads a save. Use a disposable save and one controller at a time.

Game screenshots are real Unity frames. Charts are derived from live numeric tool responses. Native fixture tests are a separate workflow.

## Setup

Install [uv](https://docs.astral.sh/uv/) and use a local OniHarness checkout containing notebook support:

```sh
uv sync --python 3.15 --extra notebook --locked
uv run --python 3.15 --extra notebook python -m ipykernel install --user \
  --name oniharness-315 --display-name "OniHarness (Python 3.15)"
uv run --python 3.15 --extra notebook jupyter lab examples/python/analyze_oni.ipynb
```

Start your installed ONI **through Steam**, with OniMcp enabled and a colony loaded and paused. Rendering needs normal graphics; a headless Null graphics device cannot supply game screenshots. Run the notebook kernel on the same machine as OniMcp.

The default endpoint is `http://127.0.0.1:8788/mcp/`. Set `ONIHARNESS_MCP_TOKEN` in the kernel environment if your server requires authentication. Never put a token in a notebook cell.

Select the project's Python 3.15 kernel. Run the cells in order. Saved output is a snapshot; it only refreshes when a cell executes.

In [ ]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from oniharness import Game, __version__

assert sys.version_info[:2] == (3, 15), "Select the Python 3.15 project kernel."
OUTPUT = Path("artifacts/notebook-analysis")
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print(f"Python {platform.python_version()} | OniHarness {__version__} | pandas {pd.__version__}")

## 1. Connect and check the public tools

Connection errors mean the game/server is unavailable. Stop and fix the connection; this notebook never substitutes fabricated colony data.

In [ ]:
game = Game()
tools = game.client.list_tools()
required = {"game_control", "navigation_control", "world_editor"}
assert required <= {tool["name"] for tool in tools}, "Required public tools are missing."
status = game.pause()
assert status["isPaused"] is True
print("Connected. Simulation pause verified.")
print("Public tools:", ", ".join(sorted(tool["name"] for tool in tools)))

## 2. Render the real game viewport

`game.frame()` requests a unique screenshot and waits for a complete PNG. The bytes are embedded in the output, so the image survives reopening the notebook. Receipt time is UTC; the cycle comes from the game.

In [ ]:
frame = game.frame()
print(frame)
display(frame)
frame.save(OUTPUT / "game-frame.png");

## 3. Collect live data with a decorator

`@game.paused` checks pause before and after this operation, including when a cell fails. The reads are sequential while simulation time is paused. Food is restricted to revealed items; it is not a count of all undiscovered resources.

Duplicant names and private save paths are excluded from the public tables. Missing measurements remain missing.

In [ ]:
@game.paused
def collect():
    return {
        "received_at_utc": datetime.now(timezone.utc).isoformat(),
        "colony": game.read("/active/colony/status.oni"),
        "food": game.read("/active/resources/food.oni"),
        "dupes": game.read("/active/dupes/index.oni"),
    }

sample = collect()
colony, food, dupes = (sample[key] for key in ("colony", "food", "dupes"))
assert colony["paused"] is True
assert food["returned"] == food["foodTypes"], "Food result is truncated; do not plot it as a complete total."
assert dupes["count"] == colony["metrics"]["dupes"], "Duplicant scope/count mismatch."
print(f"Received: {sample['received_at_utc']} | cycle: {colony['cycle']} | paused: True")
stress_values = pd.to_numeric(pd.Series([item["needs"].get("stress") for item in dupes["items"]], dtype="object"), errors="coerce")
metrics = dict(colony["metrics"])
metrics["maxStress"] = float(stress_values.max()) if stress_values.notna().any() else None
display(pd.DataFrame([metrics]).rename(columns={
    "dupes": "Duplicants", "maxStress": "Maximum stress (%)",
    "foodKcal": "Visible food (kcal)", "redAlert": "Red alert", "alerts": "Alert count",
}))

## 4. Validate and analyze food composition

The bars show stored and loose revealed food reported by OniMcp, grouped by prefab ID. This is a stock measurement in **kcal**, not a production rate or a survival forecast. Totals are rounded by the server.

In [ ]:
foods = pd.DataFrame(food["foods"])
assert not foods.empty, "No revealed food to chart. Inspect the colony instead of drawing an empty chart."
foods = foods[["prefabId", "totalCaloriesKcal", "totalMassKg", "count"]].copy()
foods.columns = ["Food", "Energy (kcal)", "Mass (kg)", "Item stacks"]
foods["Energy (kcal)"] = pd.to_numeric(foods["Energy (kcal)"], errors="raise")
assert foods["Energy (kcal)"].notna().all()
assert (foods["Energy (kcal)"] >= 0).all()
assert abs(foods["Energy (kcal)"].sum() - food["totalCaloriesKcal"]) <= 0.1 * (len(foods) + 1)
display(foods.sort_values("Energy (kcal)", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
chart = foods.sort_values("Energy (kcal)").tail(12)
ax.barh(chart["Food"], chart["Energy (kcal)"], color="#377eb8")
ax.set(title=f"Revealed food stock — cycle {colony['cycle']}", xlabel="Food energy (kcal)", ylabel="Food prefab ID")
ax.grid(axis="x", alpha=0.2)
fig.savefig(OUTPUT / "food-stock.png", dpi=160)
plt.show()

## 5. Check duplicant stress without inventing missing values

Aliases D01, D02, ... are assigned by instance ID for this sample. A missing stress value is **unknown**, not zero. Risk labels are OniMcp diagnostic hints, not a medical model or a complete colony safety assessment.

In [ ]:
rows = sorted(dupes["items"], key=lambda item: item["dupe"]["id"])
dupe_table = pd.DataFrame([
    {"Duplicant": f"D{i:02d}", "Stress (%)": item["needs"].get("stress"),
     "X": item["position"]["x"], "Y": item["position"]["y"], "Diagnostic": item["risk"]}
    for i, item in enumerate(rows, 1)
])
dupe_table["Stress (%)"] = pd.to_numeric(dupe_table["Stress (%)"], errors="coerce")
known = dupe_table.dropna(subset=["Stress (%)"])
assert known["Stress (%)"].between(0, 100).all(), "Unexpected stress unit/range."
display(dupe_table)
print(f"Stress coverage: {len(known)}/{len(dupe_table)} duplicants; unknown values are excluded from the chart.")
if not known.empty:
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    bars = ax.bar(known["Duplicant"], known["Stress (%)"], color="#4daf4a")
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.set(title=f"Duplicant stress snapshot — cycle {colony['cycle']}", ylabel="Stress (%)", ylim=(0, 105))
    ax.grid(axis="y", alpha=0.2)
    fig.savefig(OUTPUT / "duplicant-stress.png", dpi=160)
    plt.show()
else:
    (OUTPUT / "duplicant-stress.png").unlink(missing_ok=True)
    print("No known stress measurements; no stress chart was generated.")


## 6. Sample the environment where duplicants stand

These are **point samples**, one per returned duplicant. They are not a full oxygen or temperature map, and repeated positions can sample the same cell. Only valid, revealed cells are included. The cell mass is measured in kg/cell, not pressure in pascals. Element IDs are language-independent.

The scatter plot shows measured cell positions and temperatures without interpolating unsampled terrain. A hot or oxygen-poor sampled cell is a reason to inspect that location, not proof of a colony-wide problem.

In [ ]:
environment_rows = []
for i, item in enumerate(rows, 1):
    env = item["environment"]
    if env.get("valid") is not True or env.get("visible") is not True:
        continue
    environment_rows.append({
        "Duplicant": f"D{i:02d}", "X": item["position"]["x"], "Y": item["position"]["y"],
        "Element": env["element"], "Cell mass (kg)": env["massKg"],
        "Cell temperature (C)": env["temperatureC"],
    })
environment = pd.DataFrame(environment_rows)
assert not environment.empty, "No valid revealed environment samples."
assert environment[["Cell mass (kg)", "Cell temperature (C)"]].notna().all().all()
display(environment)

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
points = ax.scatter(environment["X"], environment["Y"], c=environment["Cell temperature (C)"],
                    cmap="inferno", s=150, edgecolors="white", linewidths=1)
for _, point in environment.iterrows():
    ax.annotate(point["Duplicant"], (point["X"], point["Y"]), xytext=(7, 7), textcoords="offset points")
ax.set(title=f"Cell temperature at sampled duplicant positions\nCycle {colony['cycle']}",
       xlabel="World cell X", ylabel="World cell Y")
ax.set_aspect("equal", adjustable="datalim")
ax.margins(0.25)
fig.colorbar(points, ax=ax, label="Cell temperature (C)")
fig.savefig(OUTPUT / "sampled-environment.png", dpi=160)
plt.show()

## 7. Refresh the notebook view live

This cell requests three fresh game frames and updates one output. It keeps ONI paused and stops automatically. The requested interval is added between captures; it is not a guaranteed video frame rate. Camera/UI animation can change while the simulation remains paused.

Interrupting the cell attempts a final verified pause. The saved notebook retains the final frame, not an autonomous background connection.

In [ ]:
last_frame = game.watch(frames=3, interval=0.5)
print(last_frame)
assert game.pause()["isPaused"] is True

## Checks and reproducible outputs

Export only the reviewed tables and explicit numeric metrics. Raw tool responses may contain private paths, names, or localized text and are not exported here. Do not publish your save, token, or full game logs.

In [ ]:
foods.to_csv(OUTPUT / "food-stock.csv", index=False)
dupe_table.to_csv(OUTPUT / "duplicant-status.csv", index=False)
environment.to_csv(OUTPUT / "sampled-environment.csv", index=False)
provenance = {
    "title": "Analyze Oxygen Not Included Like a Data Scientist",
    "python": platform.python_version(), "oniharness": __version__,
    "received_at_utc": sample["received_at_utc"], "cycle": colony["cycle"],
    "paused": True, "food_visible_only": food["visibleOnly"],
    "food_types": len(foods), "duplicants": len(dupe_table),
    "known_stress_values": len(known), "environment_point_samples": len(environment), "metrics": metrics,
    "scope": "one paused colony; revealed food; all returned duplicants",
}
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Saved original game PNG, food/environment charts, CSV tables and provenance JSON.")
print("Stress chart saved:", not known.empty)
print("Final pause:", game.pause()["isPaused"])

## Next steps

Ask a concrete question before collecting more data: is food composition changing, is stress concentrated in a few duplicants, or is a local oxygen pocket being missed by colony totals?

To measure change over time, collect a second sample after a separately authorized, short, guarded game step. Record both simulation time and receipt time. Repeated reads of a paused colony do not measure growth, consumption rates, performance, or causal effects.

Native mod tests use `Suite`, `@suite.scenario`, `@parametrize` and `@expect.mass_conserved`. Those tests exercise a small native fixture, not this live colony.